In [1]:
#######################################################################################
## MIT-BIH ECG: cWGAN-GP augmentation + quantitative synthetic-data validation
## + memorization checking
##
## IMPORTANT:
## The GAN augmentation block below intentionally follows the provided training script:
## - same train/validation/test split
## - same random seed
## - same generator and critic
## - same noise dimension
## - same GAN epochs
## - same critic updates and gradient penalty
## - same Adam settings
## - same per-class batch size
## - same drop_last=False behavior
## - same 47,102-sample target
##
## The validation section then reports, for every class:
##   1) RBF-MMD^2                       : distribution similarity (lower is better)
##   2) Median DTW to nearest real beat: waveform similarity (lower is better)
##   3) Median synthetic->real NN L2    : nearest-real distance
##   4) Median real->real NN L2         : natural within-class reference distance
##   5) NN distance ratio               : synth->real / real->real
##   6) Near-copy rate (%)              : memorization indicator
##   7) Exact/very-near duplicate rate  : strict memorization indicator
##   8) Mean-waveform RMSE              : mean morphology difference
##   9) Mean-waveform Pearson r         : mean morphology correlation
##
## Output:
##   GAN_Validation_Output/GAN_Synthetic_Validation_Table.csv
##   GAN_Validation_Output/GAN_Training_Diagnostics.csv
##   GAN_Validation_Output/Waveform_Class_XX.pdf
##
## NOTE:
## These measures quantify statistical/morphological similarity and memorization risk.
## They DO NOT by themselves prove clinical realism. Clinical realism requires
## independent expert/clinician assessment.
#######################################################################################

# =============================================================================
# 0. IMPORTS
# =============================================================================

import os
import random
import platform
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import pdist


# =============================================================================
# 1. REPRODUCIBILITY & DEVICE -- same seed as the provided code
# =============================================================================

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

has_gpu = torch.cuda.is_available()
has_mps = torch.backends.mps.is_built()
device = torch.device(
    "mps" if has_mps else
    "cuda" if has_gpu else
    "cpu"
)

print(f"Python Platform: {platform.platform()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Target device  : {device}")


# =============================================================================
# 2. PATHS -- same dataset path as the provided code
# =============================================================================

directory = "/scratch/user/uqabulbu/Data/"
file_name_MITBIHAR = "Data_AAMB/AR/2.MITBIHAR_Filtered_Segmented_1CH_128HZ.h5"
file_path_MITBIHAR = directory + file_name_MITBIHAR

OUTPUT_DIR = Path("GAN_Validation_Output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_CSV = OUTPUT_DIR / "GAN_Synthetic_Validation_Table.csv"
DIAGNOSTICS_CSV = OUTPUT_DIR / "GAN_Training_Diagnostics.csv"


# =============================================================================
# 3. HUMAN-READABLE CLASS LABELS
#    Numeric indices remain exactly those used by the model/GAN.
# =============================================================================

CLASS_SYMBOLS = {
    0: "N",
    1: "L",
    2: "R",
    3: "V",
    4: "/",
    5: "A",
    6: "f",
    7: "F",
    8: "!",
    9: "j",
    10: "x",
    11: "a",
    12: "E",
    13: "J",
    14: "e",
    15: "Q",
}


# =============================================================================
# 4. LOAD DATASET -- same logic as the provided code
# =============================================================================

with h5py.File(file_path_MITBIHAR, "r") as h5f:
    ecg_signals_2 = h5f["ECG_Signals"][:]
    ecg_labels_2 = h5f["ECG_Labels"][:]

combined_signals = ecg_signals_2
combined_labels = ecg_labels_2.squeeze()

print("Combined data shape :", combined_signals.shape)
print("Combined label shape:", combined_labels.shape)


# =============================================================================
# 5. TRAIN / VALIDATION / TEST SPLIT -- exactly the same split as provided
# =============================================================================

data_tensor = torch.from_numpy(combined_signals).float().unsqueeze(1)
labels_tensor = torch.from_numpy(combined_labels).long()

data_np = data_tensor.squeeze(1).numpy()
labels_np = labels_tensor.numpy()

train_val_data, test_data, train_val_labels, test_labels = train_test_split(
    data_np,
    labels_np,
    test_size=0.30,
    stratify=labels_np,
    random_state=42,
)

train_data, val_data, train_labels, val_labels = train_test_split(
    train_val_data,
    train_val_labels,
    test_size=0.10,
    stratify=train_val_labels,
    random_state=42,
)

train_data_tensor = torch.from_numpy(train_data).float().unsqueeze(1)
train_labels_tensor = torch.from_numpy(train_labels).long()

print("\nBefore GAN class distribution:")
print(np.bincount(train_labels_tensor.view(-1).numpy().astype(int)))


# =============================================================================
# 6. cWGAN-GP SETTINGS -- exactly the same values as the provided code
# =============================================================================

noise_dim = 100
gan_epochs_per_class = 50
n_critic = 5
lmda_gp = 10.0
min_samples_per_class = 47102


# =============================================================================
# 7. GENERATOR / CRITIC -- same architecture as the provided code
# =============================================================================

class Gen(nn.Module):
    def __init__(self, noise_dim=100, num_classes=16, out_dim=77):
        super().__init__()
        self.embed = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.Linear(noise_dim + num_classes, 128),
            nn.ReLU(True),
            nn.Linear(128, 256),
            nn.ReLU(True),
            nn.Linear(256, out_dim),
        )

    def forward(self, z, y):
        y_emb = self.embed(y)
        return self.net(torch.cat([z, y_emb], dim=1))


class Critic(nn.Module):
    def __init__(self, num_classes=16, in_dim=77):
        super().__init__()
        self.embed = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.Linear(in_dim + num_classes, 256),
            nn.LeakyReLU(0.2, True),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2, True),
            nn.Linear(128, 1),
        )

    def forward(self, x, y):
        y_emb = self.embed(y)
        return self.net(torch.cat([x, y_emb], dim=1))


def gradient_penalty(critic, real, fake, labels):
    alpha = torch.rand(real.size(0), 1, device=device)
    inter = (alpha * real + (1 - alpha) * fake).requires_grad_(True)

    score = critic(inter, labels)

    grad = autograd.grad(
        outputs=score,
        inputs=inter,
        grad_outputs=torch.ones_like(score),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    return ((grad.norm(2, dim=1) - 1) ** 2).mean()


# =============================================================================
# 8. SAME GAN AUGMENTATION + PER-CLASS TRAINING DIAGNOSTICS
# =============================================================================

train_data_sm = train_data_tensor.squeeze(1).numpy()
train_labels_sm = train_labels_tensor.numpy()

synthetic_batches = []
synthetic_labels = []

# Keep generated data separated by class for validation.
synthetic_by_class = {}
real_by_class = {}

training_diagnostics = []

for cls in np.unique(train_labels_sm):
    cls = int(cls)

    cls_idx = train_labels_sm == cls
    cls_count = int(cls_idx.sum())

    real_cls_np = train_data_sm[cls_idx].astype(np.float32)
    real_by_class[cls] = real_cls_np

    needed = max(0, min_samples_per_class - cls_count)

    if cls_count >= min_samples_per_class:
        training_diagnostics.append(
            {
                "Class_Index": cls,
                "Class_Symbol": CLASS_SYMBOLS.get(cls, str(cls)),
                "Real_Train_Samples": cls_count,
                "Synthetic_Requested": 0,
                "GAN_Batches_Per_Epoch": 0,
                "Generator_Updates": 0,
                "Critic_Updates": 0,
                "GAN_Trained": "Not needed",
            }
        )
        synthetic_by_class[cls] = np.empty((0, 77), dtype=np.float32)
        continue

    real_cls = torch.tensor(
        real_cls_np,
        dtype=torch.float32,
        device=device,
    )
    lbl_cls = torch.tensor(
        train_labels_sm[cls_idx],
        dtype=torch.long,
        device=device,
    )

    G = Gen(noise_dim, 16, 77).to(device)
    D = Critic(16, 77).to(device)

    opt_G = optim.Adam(
        G.parameters(),
        lr=1e-4,
        betas=(0.5, 0.9),
    )
    opt_D = optim.Adam(
        D.parameters(),
        lr=1e-4,
        betas=(0.5, 0.9),
    )

    # IMPORTANT: this is intentionally kept EXACTLY as in the provided code.
    loader_cls = DataLoader(
        TensorDataset(real_cls, lbl_cls),
        batch_size=512,
        shuffle=True,
        drop_last=False,
    )

    batches_per_epoch = len(loader_cls)
    generator_updates = gan_epochs_per_class * batches_per_epoch
    critic_updates = generator_updates * n_critic

    if batches_per_epoch == 0:
        print(
            f"\nWARNING: Class {cls} ({CLASS_SYMBOLS.get(cls, cls)}) has "
            f"{cls_count} real training samples, which is smaller than the GAN "
            f"batch size 512. Because the original code uses drop_last=False, "
            f"this class receives ZERO GAN training updates before generation."
        )

    for _ in range(gan_epochs_per_class):
        for x_real, y_real in loader_cls:

            # critic -- same as provided
            for _ in range(n_critic):
                z = torch.randn(
                    x_real.size(0),
                    noise_dim,
                    device=device,
                )

                x_fake = G(z, y_real).detach()

                loss_D = (
                    D(x_fake, y_real).mean()
                    - D(x_real, y_real).mean()
                )

                gp = gradient_penalty(
                    D,
                    x_real,
                    x_fake,
                    y_real,
                )

                loss_D = loss_D + lmda_gp * gp

                opt_D.zero_grad()
                loss_D.backward()
                opt_D.step()

            # generator -- same as provided
            z = torch.randn(
                x_real.size(0),
                noise_dim,
                device=device,
            )

            x_fake = G(z, y_real)

            loss_G = -D(x_fake, y_real).mean()

            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

    # Generate until the class is topped up -- same as provided
    class_fake_batches = []
    needed_remaining = needed

    with torch.no_grad():
        while needed_remaining > 0:
            gen_batch = min(needed_remaining, 4096)

            z = torch.randn(
                gen_batch,
                noise_dim,
                device=device,
            )

            lab = torch.full(
                (gen_batch,),
                cls,
                dtype=torch.long,
                device=device,
            )

            fake = G(z, lab).cpu().numpy().astype(np.float32)

            synthetic_batches.append(fake)
            synthetic_labels.append(
                np.full(
                    gen_batch,
                    cls,
                    dtype=train_labels_sm.dtype,
                )
            )

            class_fake_batches.append(fake)

            needed_remaining -= gen_batch

    synthetic_by_class[cls] = np.vstack(class_fake_batches)

    training_diagnostics.append(
        {
            "Class_Index": cls,
            "Class_Symbol": CLASS_SYMBOLS.get(cls, str(cls)),
            "Real_Train_Samples": cls_count,
            "Synthetic_Requested": needed,
            "GAN_Batches_Per_Epoch": batches_per_epoch,
            "Generator_Updates": generator_updates,
            "Critic_Updates": critic_updates,
            "GAN_Trained": "Yes" if generator_updates > 0 else "NO - zero updates",
        }
    )

# Save diagnostics immediately.
diagnostics_df = pd.DataFrame(training_diagnostics)
diagnostics_df.to_csv(DIAGNOSTICS_CSV, index=False)

print("\n============================================================")
print("GAN TRAINING DIAGNOSTICS")
print("============================================================")
print(diagnostics_df.to_string(index=False))
print(f"\nSaved: {DIAGNOSTICS_CSV}")


# =============================================================================
# 9. OPTIONAL MERGE -- same real + synthetic training set as the original script
# =============================================================================

if synthetic_batches:
    synth_data = np.vstack(synthetic_batches)
    synth_labels = np.hstack(synthetic_labels)

    new_train_data = np.vstack(
        [train_data_sm, synth_data]
    )

    new_train_labels = np.hstack(
        [train_labels_sm, synth_labels]
    )
else:
    synth_data = np.empty((0, 77), dtype=np.float32)
    synth_labels = np.empty((0,), dtype=train_labels_sm.dtype)

    new_train_data = train_data_sm
    new_train_labels = train_labels_sm

print("\nAfter GAN class distribution:")
print(np.bincount(new_train_labels.astype(int)))


# =============================================================================
# 10. VALIDATION SETTINGS
#
# Large synthetic classes may contain tens of thousands of samples.
# To keep MMD/DTW practical, reproducible random subsets are used.
# Nearest-neighbour memorization checks also use controlled subsets.
# =============================================================================

RNG = np.random.default_rng(42)

MMD_MAX_REAL = 500
MMD_MAX_SYNTH = 500

NN_MAX_REAL = 5000
NN_MAX_SYNTH = 5000

DTW_MAX_SYNTH = 250

# A synthetic sample is counted as a "near copy" when its nearest-real
# Euclidean distance is <= the 5th percentile of natural real-to-real
# nearest-neighbour distances for that class.
NEAR_COPY_REAL_NN_PERCENTILE = 5.0

# Very strict numerical duplicate threshold.
EXACT_DUPLICATE_L2_THRESHOLD = 1e-6


# =============================================================================
# 11. HELPER FUNCTIONS
# =============================================================================

def random_subset(x, max_n, rng):
    """Return all samples if small; otherwise a reproducible random subset."""
    x = np.asarray(x)

    if len(x) <= max_n:
        return x

    idx = rng.choice(
        len(x),
        size=max_n,
        replace=False,
    )
    return x[idx]


def rbf_mmd2(real, synth, rng):
    """
    Biased RBF Maximum Mean Discrepancy squared.

    Lower is better.
    0 means the two sampled distributions are indistinguishable under
    the selected RBF kernel.

    The RBF bandwidth is selected using the median-distance heuristic
    on the combined real/synthetic subset.
    """

    x = random_subset(
        real,
        MMD_MAX_REAL,
        rng,
    ).astype(np.float64)

    y = random_subset(
        synth,
        MMD_MAX_SYNTH,
        rng,
    ).astype(np.float64)

    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan

    combined = np.vstack([x, y])

    d = pdist(
        combined,
        metric="euclidean",
    )

    positive_d = d[d > 0]

    if len(positive_d) == 0:
        sigma = 1.0
    else:
        sigma = np.median(positive_d)

    if sigma <= 0 or not np.isfinite(sigma):
        sigma = 1.0

    gamma = 1.0 / (2.0 * sigma * sigma)

    d_xx = pairwise_distances(
        x,
        x,
        metric="sqeuclidean",
    )

    d_yy = pairwise_distances(
        y,
        y,
        metric="sqeuclidean",
    )

    d_xy = pairwise_distances(
        x,
        y,
        metric="sqeuclidean",
    )

    k_xx = np.exp(-gamma * d_xx)
    k_yy = np.exp(-gamma * d_yy)
    k_xy = np.exp(-gamma * d_xy)

    mmd2 = (
        k_xx.mean()
        + k_yy.mean()
        - 2.0 * k_xy.mean()
    )

    # Small numerical negatives can occur.
    mmd2 = max(0.0, float(mmd2))

    return mmd2, float(sigma)


def dtw_distance_per_sample(x, y):
    """
    DTW using absolute pointwise cost.

    Both ECG segments have length 77.
    The final accumulated cost is divided by max(len(x), len(y))
    so the reported number is easier to compare across runs.
    """

    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    n = len(x)
    m = len(y)

    prev = np.full(m + 1, np.inf)
    prev[0] = 0.0

    for i in range(1, n + 1):
        curr = np.full(m + 1, np.inf)

        for j in range(1, m + 1):
            cost = abs(x[i - 1] - y[j - 1])

            curr[j] = cost + min(
                prev[j],      # insertion
                curr[j - 1],  # deletion
                prev[j - 1],  # match
            )

        prev = curr

    return float(prev[m] / max(n, m))


def nearest_neighbor_statistics(real, synth, rng):
    """
    Memorization diagnostics.

    Returns:
      median synthetic->real NN L2
      median real->real NN L2
      near-copy threshold
      near-copy rate
      exact/very-near duplicate rate
      selected synthetic NN distances
      nearest real indices for selected synthetic signals
      real subset used for NN analysis
      synthetic subset used for NN analysis
    """

    real_nn = random_subset(
        real,
        NN_MAX_REAL,
        rng,
    ).astype(np.float64)

    synth_nn = random_subset(
        synth,
        NN_MAX_SYNTH,
        rng,
    ).astype(np.float64)

    if len(real_nn) == 0 or len(synth_nn) == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.array([]),
            np.array([], dtype=int),
            real_nn,
            synth_nn,
        )

    # -------------------------------------------------------------
    # Synthetic -> nearest real
    # -------------------------------------------------------------
    nbr_real = NearestNeighbors(
        n_neighbors=1,
        metric="euclidean",
    )

    nbr_real.fit(real_nn)

    syn_distances, syn_indices = nbr_real.kneighbors(
        synth_nn,
        return_distance=True,
    )

    syn_distances = syn_distances[:, 0]
    syn_indices = syn_indices[:, 0]

    median_syn_real = float(
        np.median(syn_distances)
    )

    # -------------------------------------------------------------
    # Real -> nearest *other* real
    # -------------------------------------------------------------
    if len(real_nn) >= 2:
        nbr_rr = NearestNeighbors(
            n_neighbors=2,
            metric="euclidean",
        )

        nbr_rr.fit(real_nn)

        rr_distances, _ = nbr_rr.kneighbors(
            real_nn,
            return_distance=True,
        )

        # first neighbour is self; second is the nearest other real
        rr_nn = rr_distances[:, 1]

        median_real_real = float(
            np.median(rr_nn)
        )

        near_copy_threshold = float(
            np.percentile(
                rr_nn,
                NEAR_COPY_REAL_NN_PERCENTILE,
            )
        )

        near_copy_rate = float(
            100.0
            * np.mean(
                syn_distances <= near_copy_threshold
            )
        )
    else:
        median_real_real = np.nan
        near_copy_threshold = np.nan
        near_copy_rate = np.nan

    exact_duplicate_rate = float(
        100.0
        * np.mean(
            syn_distances <= EXACT_DUPLICATE_L2_THRESHOLD
        )
    )

    return (
        median_syn_real,
        median_real_real,
        near_copy_threshold,
        near_copy_rate,
        exact_duplicate_rate,
        syn_distances,
        syn_indices,
        real_nn,
        synth_nn,
    )


def median_dtw_to_nearest_real(
    real_nn,
    synth_nn,
    synth_nn_indices,
    rng,
):
    """
    Compute DTW only for a reproducible subset of synthetic beats,
    pairing each synthetic beat with its Euclidean nearest real beat.
    """

    if len(synth_nn) == 0 or len(real_nn) == 0:
        return np.nan

    n = min(
        DTW_MAX_SYNTH,
        len(synth_nn),
    )

    if len(synth_nn) <= n:
        chosen = np.arange(
            len(synth_nn)
        )
    else:
        chosen = rng.choice(
            len(synth_nn),
            size=n,
            replace=False,
        )

    dtw_values = []

    for idx in chosen:
        fake = synth_nn[idx]
        real_match = real_nn[
            synth_nn_indices[idx]
        ]

        dtw_values.append(
            dtw_distance_per_sample(
                fake,
                real_match,
            )
        )

    return float(
        np.median(dtw_values)
    )


def mean_waveform_metrics(real, synth):
    """
    Compare class-average real and synthetic ECG morphology.
    """

    if len(real) == 0 or len(synth) == 0:
        return np.nan, np.nan

    mean_real = np.mean(
        real,
        axis=0,
    )

    mean_synth = np.mean(
        synth,
        axis=0,
    )

    rmse = float(
        np.sqrt(
            np.mean(
                (mean_real - mean_synth) ** 2
            )
        )
    )

    real_std = np.std(mean_real)
    synth_std = np.std(mean_synth)

    if real_std == 0 or synth_std == 0:
        corr = np.nan
    else:
        corr = float(
            np.corrcoef(
                mean_real,
                mean_synth,
            )[0, 1]
        )

    return rmse, corr


def safe_filename_token(value):
    """
    Convert an annotation symbol into a file-system-safe token.

    This is necessary because the MIT-BIH paced-beat symbol "/" is treated
    as a directory separator on Linux/macOS/Windows path handling.
    """
    symbol_names = {
        "/": "paced",
        "!": "flutter",
    }
    value = symbol_names.get(str(value), str(value))

    safe = []
    for ch in value:
        if ch.isalnum() or ch in ("-", "_"):
            safe.append(ch)
        else:
            safe.append("_")

    token = "".join(safe).strip("_")
    return token if token else "class"


def save_waveform_plot(
    cls,
    real,
    synth,
):
    """
    Save real-vs-synthetic mean waveform with ±1 SD bands.
    """

    if len(real) == 0 or len(synth) == 0:
        return

    # Re-create the output folder here as an extra safeguard.
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    real_plot = random_subset(
        real,
        min(5000, len(real)),
        RNG,
    )

    synth_plot = random_subset(
        synth,
        min(5000, len(synth)),
        RNG,
    )

    x = np.arange(real.shape[1])

    mean_real = real_plot.mean(axis=0)
    std_real = real_plot.std(axis=0)

    mean_synth = synth_plot.mean(axis=0)
    std_synth = synth_plot.std(axis=0)

    fig, ax = plt.subplots(
        figsize=(7.2, 4.2)
    )

    ax.plot(
        x,
        mean_real,
        linewidth=2,
        label="Real mean",
    )

    ax.fill_between(
        x,
        mean_real - std_real,
        mean_real + std_real,
        alpha=0.15,
    )

    ax.plot(
        x,
        mean_synth,
        linewidth=2,
        linestyle="--",
        label="Synthetic mean",
    )

    ax.fill_between(
        x,
        mean_synth - std_synth,
        mean_synth + std_synth,
        alpha=0.15,
    )

    ax.set_xlabel("Sample index")
    ax.set_ylabel("ECG amplitude")

    class_symbol = CLASS_SYMBOLS.get(cls, str(cls))
    ax.set_title(
        f"Class {cls} ({class_symbol}): Real vs Synthetic ECG"
    )

    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    safe_symbol = safe_filename_token(class_symbol)
    base_name = f"Waveform_Class_{cls:02d}_{safe_symbol}"

    fig.savefig(
        OUTPUT_DIR / f"{base_name}.pdf",
        bbox_inches="tight",
        format="pdf",
    )

    fig.savefig(
        OUTPUT_DIR / f"{base_name}.png",
        dpi=300,
        bbox_inches="tight",
        format="png",
    )

    plt.close(fig)


# =============================================================================
# 12. RUN QUANTITATIVE VALIDATION + MEMORIZATION CHECKS
# =============================================================================

validation_rows = []

print("\n============================================================")
print("SYNTHETIC ECG QUANTITATIVE VALIDATION")
print("============================================================")

for cls in range(16):
    real = real_by_class.get(
        cls,
        np.empty((0, 77), dtype=np.float32),
    )

    synth = synthetic_by_class.get(
        cls,
        np.empty((0, 77), dtype=np.float32),
    )

    real_count = len(real)
    synth_count = len(synth)

    diag_row = diagnostics_df[
        diagnostics_df["Class_Index"] == cls
    ]

    if len(diag_row) > 0:
        batches_per_epoch = int(
            diag_row.iloc[0]["GAN_Batches_Per_Epoch"]
        )
        generator_updates = int(
            diag_row.iloc[0]["Generator_Updates"]
        )
        critic_updates = int(
            diag_row.iloc[0]["Critic_Updates"]
        )
        gan_trained = diag_row.iloc[0]["GAN_Trained"]
    else:
        batches_per_epoch = 0
        generator_updates = 0
        critic_updates = 0
        gan_trained = "Unknown"

    if synth_count == 0:
        validation_rows.append(
            {
                "Class_Index": cls,
                "Class_Symbol": CLASS_SYMBOLS.get(cls, str(cls)),
                "Real_Train_Samples": real_count,
                "Synthetic_Samples": synth_count,
                "Synthetic_to_Real_Ratio": 0.0,
                "GAN_Batches_Per_Epoch": batches_per_epoch,
                "Generator_Updates": generator_updates,
                "Critic_Updates": critic_updates,
                "GAN_Trained": gan_trained,
                "MMD_RBF2": np.nan,
                "MMD_RBF_Sigma": np.nan,
                "Median_DTW_to_Nearest_Real": np.nan,
                "Median_NN_L2_Synth_to_Real": np.nan,
                "Median_NN_L2_Real_to_Real": np.nan,
                "NN_Distance_Ratio": np.nan,
                "Near_Copy_Threshold_L2": np.nan,
                "Near_Copy_Rate_pct": np.nan,
                "Exact_Duplicate_Rate_pct": np.nan,
                "Mean_Waveform_RMSE": np.nan,
                "Mean_Waveform_Pearson_r": np.nan,
            }
        )

        print(
            f"Class {cls:2d} ({CLASS_SYMBOLS.get(cls, cls)}): "
            f"no synthetic samples required."
        )
        continue

    # ---------------------------------------------------------
    # MMD
    # ---------------------------------------------------------
    mmd2, sigma = rbf_mmd2(
        real,
        synth,
        RNG,
    )

    # ---------------------------------------------------------
    # Memorization / nearest-neighbour statistics
    # ---------------------------------------------------------
    (
        median_syn_real,
        median_real_real,
        near_copy_threshold,
        near_copy_rate,
        exact_duplicate_rate,
        syn_distances,
        syn_indices,
        real_nn,
        synth_nn,
    ) = nearest_neighbor_statistics(
        real,
        synth,
        RNG,
    )

    if (
        np.isfinite(median_real_real)
        and median_real_real > 0
    ):
        nn_ratio = (
            median_syn_real
            / median_real_real
        )
    else:
        nn_ratio = np.nan

    # ---------------------------------------------------------
    # DTW to nearest real
    # ---------------------------------------------------------
    median_dtw = median_dtw_to_nearest_real(
        real_nn,
        synth_nn,
        syn_indices,
        RNG,
    )

    # ---------------------------------------------------------
    # Mean morphology
    # ---------------------------------------------------------
    mean_rmse, mean_corr = mean_waveform_metrics(
        real,
        synth,
    )

    # ---------------------------------------------------------
    # Plot
    # ---------------------------------------------------------
    save_waveform_plot(
        cls,
        real,
        synth,
    )

    validation_rows.append(
        {
            "Class_Index": cls,
            "Class_Symbol": CLASS_SYMBOLS.get(cls, str(cls)),
            "Real_Train_Samples": real_count,
            "Synthetic_Samples": synth_count,
            "Synthetic_to_Real_Ratio": synth_count / real_count if real_count > 0 else np.nan,
            "GAN_Batches_Per_Epoch": batches_per_epoch,
            "Generator_Updates": generator_updates,
            "Critic_Updates": critic_updates,
            "GAN_Trained": gan_trained,
            "MMD_RBF2": mmd2,
            "MMD_RBF_Sigma": sigma,
            "Median_DTW_to_Nearest_Real": median_dtw,
            "Median_NN_L2_Synth_to_Real": median_syn_real,
            "Median_NN_L2_Real_to_Real": median_real_real,
            "NN_Distance_Ratio": nn_ratio,
            "Near_Copy_Threshold_L2": near_copy_threshold,
            "Near_Copy_Rate_pct": near_copy_rate,
            "Exact_Duplicate_Rate_pct": exact_duplicate_rate,
            "Mean_Waveform_RMSE": mean_rmse,
            "Mean_Waveform_Pearson_r": mean_corr,
        }
    )

    print(
        f"Class {cls:2d} ({CLASS_SYMBOLS.get(cls, cls)}): "
        f"real={real_count:5d}, "
        f"synth={synth_count:5d}, "
        f"MMD^2={mmd2:.6f}, "
        f"DTW={median_dtw:.6f}, "
        f"NNsyn-real={median_syn_real:.6f}, "
        f"NNreal-real={median_real_real:.6f}, "
        f"near-copy={near_copy_rate:.2f}%, "
        f"exact-dup={exact_duplicate_rate:.4f}%"
    )


# =============================================================================
# 13. SAVE COMPLETE TABLE
# =============================================================================

validation_df = pd.DataFrame(validation_rows)

validation_df.to_csv(
    VALIDATION_CSV,
    index=False,
)

print("\n============================================================")
print("FINAL VALIDATION TABLE")
print("============================================================")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print(
    validation_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print(f"\nSaved final table to: {VALIDATION_CSV}")


# =============================================================================
# 14. SMALL REVIEWER-READY TABLE
# =============================================================================

reviewer_columns = [
    "Class_Symbol",
    "Real_Train_Samples",
    "Synthetic_Samples",
    "Synthetic_to_Real_Ratio",
    "MMD_RBF2",
    "Median_DTW_to_Nearest_Real",
    "Median_NN_L2_Synth_to_Real",
    "Median_NN_L2_Real_to_Real",
    "NN_Distance_Ratio",
    "Near_Copy_Rate_pct",
    "Exact_Duplicate_Rate_pct",
    "Mean_Waveform_RMSE",
    "Mean_Waveform_Pearson_r",
]

reviewer_table = validation_df[
    reviewer_columns
].copy()

reviewer_path = (
    OUTPUT_DIR
    / "GAN_Reviewer_Ready_Validation_Table.csv"
)

reviewer_table.to_csv(
    reviewer_path,
    index=False,
)

print(
    f"Saved reviewer-ready table to: "
    f"{reviewer_path}"
)


# =============================================================================
# 15. INTERPRETATION GUIDE PRINTED WITH THE RESULTS
# =============================================================================

print(
    r"""
============================================================
HOW TO INTERPRET THE OUTPUT
============================================================

MMD_RBF2
    Lower = closer real/synthetic class distributions.
    Use class-wise values comparatively; there is no universal clinical cutoff.

Median_DTW_to_Nearest_Real
    Lower = synthetic waveforms are morphologically closer to real beats.
    Very low DTW alone is NOT automatically good because memorized samples
    can also have very low DTW.

Median_NN_L2_Synth_to_Real
    Median Euclidean distance from a synthetic ECG to its nearest real ECG.

Median_NN_L2_Real_to_Real
    Natural reference: median distance from each real ECG to the nearest
    different real ECG within the same class.

NN_Distance_Ratio
    = median synthetic->real NN distance / median real->real NN distance.

    Rough interpretation:
        around 1 : synthetic samples have nearest-real distances similar to
                   natural real-to-real diversity.
        << 1     : synthetic beats may be unusually close to training beats;
                   investigate possible memorization.
        >> 1     : synthetic beats may be too far from the real class manifold.

Near_Copy_Rate_pct
    Percentage of tested synthetic beats whose nearest-real distance is at
    or below the 5th percentile of real-to-real nearest-neighbour distances.
    A high value is a memorization warning, not a definitive proof.

Exact_Duplicate_Rate_pct
    Percentage of synthetic beats with nearest-real L2 <= 1e-6.
    Ideally this should be 0%.

Mean_Waveform_RMSE
    Lower = closer average waveform.

Mean_Waveform_Pearson_r
    Closer to +1 = more similar average waveform shape.

IMPORTANT
    These statistics can support claims of statistical/morphological similarity,
    but they cannot establish clinical realism. Clinical plausibility should be
    assessed independently by qualified ECG experts.

============================================================
"""
)

# Explicitly identify classes that received zero GAN updates.
zero_update = validation_df[
    (validation_df["Synthetic_Samples"] > 0)
    & (validation_df["Generator_Updates"] == 0)
]

if len(zero_update) > 0:
    print(
        "\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!"
    )
    print(
        "CRITICAL WARNING: The following augmented classes received ZERO "
        "GAN training updates under the ORIGINAL augmentation settings:"
    )

    print(
        zero_update[
            [
                "Class_Index",
                "Class_Symbol",
                "Real_Train_Samples",
                "Synthetic_Samples",
            ]
        ].to_string(index=False)
    )

    print(
        "\nReason: the original DataLoader uses batch_size=512 and "
        "drop_last=False. Any class with fewer than 512 real training "
        "samples produces 1 batches. Its generator therefore remains "
        "at random initialization before synthetic ECG generation."
    )

    print(
        "\nDo not describe synthetic ECGs from these classes as GAN-learned "
        "or clinically realistic unless the GAN training procedure is corrected "
        "and the downstream model is retrained."
    )

    print(
        "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n"
    )


Python Platform: Linux-5.14.0-687.29.1.el9_8.x86_64-x86_64-with-glibc2.34
PyTorch Version: 2.7.1+cu126
Target device  : cuda
Combined data shape : (105708, 77)
Combined label shape: (105708,)

Before GAN class distribution:
[47102  5085  4571  4487  2281  1604   164   505   298   144   121    94
    67    52    10    10]


/home/uqabulbu/.local/lib/python3.10/site-packages/torch/autograd/graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



GAN TRAINING DIAGNOSTICS
 Class_Index Class_Symbol  Real_Train_Samples  Synthetic_Requested  GAN_Batches_Per_Epoch  Generator_Updates  Critic_Updates GAN_Trained
           0            N               47102                    0                      0                  0               0  Not needed
           1            L                5085                42017                     10                500            2500         Yes
           2            R                4571                42531                      9                450            2250         Yes
           3            V                4487                42615                      9                450            2250         Yes
           4            /                2281                44821                      5                250            1250         Yes
           5            A                1604                45498                      4                200            1000         Yes
           6   

OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detec

OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detec

...

Class 11 (a): real=   94, synth=47008, MMD^2=0.658540, DTW=0.156047, NNsyn-real=1.773416, NNreal-real=0.446316, near-copy=0.00%, exact-dup=0.0000%


OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detec

Class 12 (E): real=   67, synth=47035, MMD^2=1.159076, DTW=0.134402, NNsyn-real=1.750077, NNreal-real=0.215086, near-copy=0.00%, exact-dup=0.0000%


OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detec

Class 13 (J): real=   52, synth=47050, MMD^2=1.002079, DTW=0.154115, NNsyn-real=1.851721, NNreal-real=0.317686, near-copy=0.00%, exact-dup=0.0000%
Class 14 (e): real=   10, synth=47092, MMD^2=1.303022, DTW=0.184672, NNsyn-real=2.993465, NNreal-real=0.537596, near-copy=0.00%, exact-dup=0.0000%
Class 15 (Q): real=   10, synth=47092, MMD^2=0.703451, DTW=0.255139, NNsyn-real=3.117089, NNreal-real=4.737717, near-copy=0.04%, exact-dup=0.0000%

FINAL VALIDATION TABLE
 Class_Index Class_Symbol  Real_Train_Samples  Synthetic_Samples  Synthetic_to_Real_Ratio  GAN_Batches_Per_Epoch  Generator_Updates  Critic_Updates GAN_Trained  MMD_RBF2  MMD_RBF_Sigma  Median_DTW_to_Nearest_Real  Median_NN_L2_Synth_to_Real  Median_NN_L2_Real_to_Real  NN_Distance_Ratio  Near_Copy_Threshold_L2  Near_Copy_Rate_pct  Exact_Duplicate_Rate_pct  Mean_Waveform_RMSE  Mean_Waveform_Pearson_r
           0            N               47102                  0                 0.000000                      0                  0  